# Tables 13 and 14

The fourteen models and the four encoders with their layer counts and widths, the 24 text carrier templates, and the spoken carriers.

Inputs: the registry in `src/models.py`, the carrier constants in `src/wordtext.py` and `src/wordspeech.py`, and the Hugging Face configurations that `scripts/features/embed.py`, `scripts/features/word_text.py` and `scripts/features/word_speech.py` download.

In [ ]:
from collections import Counter

import pandas as pd
from transformers import AutoConfig

from src import models
from src.features import speech, text
from src.paper.style import setup_notebook

setup_notebook()

## Table 13

Layers and width come from each checkpoint's cached configuration, read from the vision configuration where the checkpoint has one.

In [ ]:
rows = []
for key in (*models.DEFAULT_SWEEP, *models.ENCODERS):
    if key in models.MODELS:
        spec = models.MODELS[key]
        identifier = spec.repo + (" (configuration only)" if spec.kind == "random" else "")
    else:
        identifier = models.ENCODERS[key][0]
    layers, width = models.layers_and_width(key)
    rows.append({"Model": models.PRETTY[key],
                 "Checkpoint or configuration identifier": identifier,
                 "Layers": layers, "Width": width})
table13 = pd.DataFrame(rows).set_index("Model")
table13

The DINOv3, Web-DINO, DINOv2 and MAE models are self-supervised.

In [ ]:
sweep = [models.MODELS[key] for key in models.DEFAULT_SWEEP]
print(f"models: {len(sweep)}")
for family, n in Counter(spec.family for spec in sweep).items():
    print(f"  {family}: {n}")
print(f"self-supervised: {sum(spec.kind in ('dino', 'mae') for spec in sweep)}")
print(f"encoders: {len(models.ENCODERS)} ({len(text.TEXT_MODELS)} text, "
      f"{len(speech.SPEECH_MODELS)} speech)")

A speech vector concatenates relative-position bins of the w2v-BERT 2.0 width, and the untrained ViT-B takes its architecture from the listed configuration.

In [ ]:
speech_width = table13.loc[models.PRETTY["w2v-bert-2.0-frozen"], "Width"]
print(f"speech vector: {speech.BINS} bins x {speech_width} = "
      f"{speech.BINS * speech_width:,} dimensions")
untrained = AutoConfig.from_pretrained(models.MODELS["vit-random-b"].repo, local_files_only=True)
print(f"untrained ViT-B: {untrained.num_attention_heads} attention heads, "
      f"{untrained.patch_size} x {untrained.patch_size} patches")

## Table 14

The first variant of each text carrier template, with w standing for the word.

In [ ]:
table14 = pd.DataFrame(
    {"Template": [variants[0].format(w="w") for _, variants in text.TEMPLATE_FRAMES]},
    index=pd.RangeIndex(1, len(text.TEMPLATE_FRAMES) + 1, name="#"))
table14

In [ ]:
print(f"templates: {len(text.TEMPLATE_FRAMES)}")
print(f"variants per template: {text.VARIANTS}")
print(f"carriers per word: {len(text.CARRIERS)}")
print("variants of template 4: "
      + " | ".join(variant.format(w="w") for variant in text.TEMPLATE_FRAMES[3][1]))

## Spoken carriers

Each word is spoken in every carrier sentence, voice and speaking rate.

In [ ]:
print(f"synthesizer: {speech.TTS_REPO}")
print("carrier sentences: " + " | ".join(carrier.format(w="w") for carrier in speech.CARRIERS))
print(f"carriers: {len(speech.CARRIERS)}")
print(f"voices: {len(speech.SPEAKERS)} (VCTK speakers {', '.join(map(str, speech.SPEAKERS))})")
print(f"speaking rates: {len(speech.RATES)} ({', '.join(f'{rate:.1f}' for rate in speech.RATES)})")
print(f"utterances per word: {len(speech.grid())}")
print(f"relative-position bins: {speech.BINS}")